# 🧵 スパースモデリングとグループラッソ入門

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Murakami-Shoya/study-notebooks/blob/main/notebooks/2026-10-20_group-lasso.ipynb)

**出典**: [Notion教材ページ](https://app.notion.com/p/3f43834a206d81d98a78c761f713daea)

**日付**: 2026-10-20

In [ ]:
# セットアップ（ローカル・Colab両対応）
import sys

IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    # Colabには標準で必要なパッケージが揃っている
    pass

print(f"環境: {'Google Colab' if IN_COLAB else 'ローカル'}")

## 要点（3行以内）

- L1正則化（Lasso）は係数をちょうど0にすることで特徴選択を実現する凸最適化手法
- グループラッソは特徴のグループを丸ごと選択・削除でき，ECoGの電極選択や帯域選択に有用
- 近接勾配法（ISTA/FISTA）でブロック軟閾値を用いて効率的に解ける

## 1. 理論：スパースモデリングとは

**スパースモデリング**: 応答を説明するのに必要な係数は少数で，残りは0と仮定して推定する考え方．予測と特徴選択を同時に行う．

**L0からL1への緩和**:
- 理想: $\|w\|_0$（非ゼロの個数）を罰するが組合せ探索になり非凸
- 緩和: $\|w\|_1$（絶対値和）に置き換える．L1は凸で効率的に解けながら0を生む性質を持つ

**Lasso**（Tibshirani 1996）:
$$
\min_w \; \frac{1}{2n}\|y - Xw\|_2^2 + \lambda \|w\|_1
$$

**なぜ0になるか**:
- 幾何: L1制約領域はひし形で角（軸上＝係数が0）を持つ．損失の等高線が角で接しやすい
- 式: 軟閾値関数 $S_\lambda(z) = \mathrm{sign}(z)\max(|z| - \lambda, 0)$ により，$|z| \le \lambda$ の係数は0になる

**グループラッソ**（Yuan & Lin 2006）:
$$
\min_w \; \frac{1}{2n}\|y - Xw\|_2^2 + \lambda \sum_{g=1}^{G} \sqrt{p_g}\,\|w_g\|_2
$$

- 特徴をグループに分け，**グループごと残すか0にする**
- $\sqrt{p_g}$: グループサイズによる不公平を打ち消す重み
- ブロック軟閾値: $w_g \leftarrow \left(1 - \frac{\lambda\sqrt{p_g}}{\|z_g\|_2}\right)_+ z_g$
- グループ内はL2なので個々の係数は0にならない（グループ単位でのみ0）

## 2. コードで確認：LassoとRidgeの係数経路

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import Lasso, Ridge
from sklearn.preprocessing import StandardScaler

# 合成データ生成：10特徴のうち3つだけが効く
rng = np.random.default_rng(42)
n, p = 100, 10
X = rng.standard_normal((n, p))
w_true = np.zeros(p)
w_true[[0, 2, 5]] = [2.0, -1.5, 1.0]
y = X @ w_true + 0.5 * rng.standard_normal(n)

# 標準化
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# 正則化経路の計算
alphas = np.logspace(-3, 1, 50)
lasso_coefs = []
ridge_coefs = []

for alpha in alphas:
    lasso = Lasso(alpha=alpha, max_iter=10000).fit(X_scaled, y)
    ridge = Ridge(alpha=alpha).fit(X_scaled, y)
    lasso_coefs.append(lasso.coef_)
    ridge_coefs.append(ridge.coef_)

lasso_coefs = np.array(lasso_coefs)
ridge_coefs = np.array(ridge_coefs)

# 可視化
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Lasso: 係数が0になる
for i in range(p):
    axes[0].plot(alphas, lasso_coefs[:, i], label=f'w{i}' if i in [0, 2, 5] else None)
axes[0].set_xscale('log')
axes[0].axhline(0, color='k', linestyle='--', linewidth=0.5)
axes[0].set_xlabel('λ (alpha)')
axes[0].set_ylabel('係数')
axes[0].set_title('Lasso: 係数がちょうど0になる')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Ridge: 0に近づくが0にはならない
for i in range(p):
    axes[1].plot(alphas, ridge_coefs[:, i], label=f'w{i}' if i in [0, 2, 5] else None)
axes[1].set_xscale('log')
axes[1].axhline(0, color='k', linestyle='--', linewidth=0.5)
axes[1].set_xlabel('λ (alpha)')
axes[1].set_ylabel('係数')
axes[1].set_title('Ridge: 縮むが0にはならない')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"\nLasso (α=0.1): 非ゼロ係数 {np.sum(np.abs(Lasso(0.1).fit(X_scaled, y).coef_) > 1e-5)}/{p}")
print(f"Ridge (α=0.1): 非ゼロ係数 {np.sum(np.abs(Ridge(0.1).fit(X_scaled, y).coef_) > 1e-5)}/{p}")

## 3. グループラッソの実装（ISTA近接勾配法）

**アルゴリズム**:
1. 勾配ステップ: $z \leftarrow w - \eta \nabla f(w)$，ただし $\eta = 1/L$（$L$は$X^\top X/n$の最大固有値）
2. ブロック軟閾値: 各グループ$g$について $w_g \leftarrow \left(1 - \frac{\lambda\sqrt{p_g}}{\|z_g\|_2}\right)_+ z_g$

**合成データ**: 10電極×4帯域（特徴），電極0と1だけが効く

In [ ]:
def group_lasso_ista(X, y, groups, lam, n_iter=500, tol=1e-6):
    """
    グループラッソをISTA（近接勾配法）で解く
    
    Parameters
    ----------
    X : ndarray (n, p)
        特徴量行列（標準化済み推奨）
    y : ndarray (n,)
        目的変数
    groups : ndarray (p,)
        各特徴のグループID
    lam : float
        正則化パラメータ
    n_iter : int
        最大反復数
    tol : float
        収束判定の閾値
    
    Returns
    -------
    w : ndarray (p,)
        推定された係数
    """
    n, p = X.shape
    w = np.zeros(p)
    
    # リプシッツ定数（勾配の最大変化率）
    L = np.linalg.norm(X, ord=2) ** 2 / n
    eta = 1.0 / L
    
    for iteration in range(n_iter):
        w_old = w.copy()
        
        # 勾配ステップ: z = w - η * ∇f(w)
        residual = X @ w - y
        grad = X.T @ residual / n
        z = w - eta * grad
        
        # ブロック軟閾値（各グループごと）
        for g in np.unique(groups):
            idx = groups == g
            p_g = idx.sum()
            z_g = z[idx]
            
            norm_z_g = np.linalg.norm(z_g)
            threshold = lam * np.sqrt(p_g) * eta
            
            if norm_z_g > threshold:
                w[idx] = (1 - threshold / norm_z_g) * z_g
            else:
                w[idx] = 0.0
        
        # 収束判定
        if np.linalg.norm(w - w_old) < tol:
            break
    
    return w


# 合成データ：10電極 × 4帯域
rng = np.random.default_rng(0)
n_samples = 200
n_electrodes = 10
n_bands = 4  # 例: delta, theta, alpha, beta
p = n_electrodes * n_bands

X = rng.standard_normal((n_samples, p))

# 真の係数：電極0と1だけが効く
w_true = np.zeros(p)
w_true[0:n_bands] = [1.0, -0.5, 0.8, 0.3]        # 電極0
w_true[n_bands:2*n_bands] = [0.6, 0.6, -0.4, 0.2]  # 電極1

y = X @ w_true + 0.5 * rng.standard_normal(n_samples)

# グループID（電極ごと）
groups = np.repeat(np.arange(n_electrodes), n_bands)

# 標準化
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

print(f"データ: {n_samples}サンプル, {n_electrodes}電極 × {n_bands}帯域 = {p}特徴")
print(f"真に効く電極: 0, 1")

## 4. LassoとGroup Lassoの比較

In [ ]:
# Lasso
lasso = Lasso(alpha=0.1, max_iter=10000).fit(X_scaled, y)

# Group Lasso
w_glasso = group_lasso_ista(X_scaled, y, groups, lam=0.1)

# 残った電極を抽出
def active_groups(w, groups, threshold=1e-8):
    active = []
    for g in np.unique(groups):
        if np.any(np.abs(w[groups == g]) > threshold):
            active.append(g)
    return active

lasso_active = active_groups(lasso.coef_, groups)
glasso_active = active_groups(w_glasso, groups)

print(f"\nLasso:")
print(f"  非ゼロ係数: {np.sum(np.abs(lasso.coef_) > 1e-8)}/{p}")
print(f"  残った電極: {lasso_active}")

print(f"\nGroup Lasso:")
print(f"  非ゼロ係数: {np.sum(np.abs(w_glasso) > 1e-8)}/{p}")
print(f"  残った電極: {glasso_active}")

print(f"\n→ Group Lassoは電極単位で選択し，真の電極0, 1を正しく抽出")

## 5. 係数の経路（正則化パラメータλを変化）

In [ ]:
# λの経路
lambda_max_lasso = np.max(np.abs(X_scaled.T @ y)) / n_samples
lambda_max_glasso = np.max([np.linalg.norm(X_scaled[:, groups == g].T @ y) / 
                             (n_samples * np.sqrt(n_bands)) 
                             for g in range(n_electrodes)])

lambdas_lasso = np.logspace(np.log10(lambda_max_lasso), -3, 30)
lambdas_glasso = np.logspace(np.log10(lambda_max_glasso), -3, 30)

# Lassoの経路
lasso_path = []
for lam in lambdas_lasso:
    w = Lasso(alpha=lam, max_iter=10000).fit(X_scaled, y).coef_
    lasso_path.append(w)
lasso_path = np.array(lasso_path)

# Group Lassoの経路
glasso_path = []
for lam in lambdas_glasso:
    w = group_lasso_ista(X_scaled, y, groups, lam=lam)
    glasso_path.append(w)
glasso_path = np.array(glasso_path)

# 可視化
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Lasso経路（電極ごとに色分け）
colors = plt.cm.tab10(np.arange(n_electrodes))
for i in range(p):
    electrode = i // n_bands
    axes[0].plot(lambdas_lasso, lasso_path[:, i], 
                color=colors[electrode], alpha=0.7, linewidth=1)

axes[0].set_xscale('log')
axes[0].axhline(0, color='k', linestyle='--', linewidth=0.5)
axes[0].set_xlabel('λ')
axes[0].set_ylabel('係数')
axes[0].set_title('Lasso: 係数がばらばらに入る')
axes[0].invert_xaxis()
axes[0].grid(True, alpha=0.3)

# Group Lasso経路（電極ごとに色分け，同じ電極の係数は同時に入る）
for i in range(p):
    electrode = i // n_bands
    axes[1].plot(lambdas_glasso, glasso_path[:, i], 
                color=colors[electrode], alpha=0.7, linewidth=1,
                label=f'電極{electrode}' if i % n_bands == 0 and electrode < 3 else None)

axes[1].set_xscale('log')
axes[1].axhline(0, color='k', linestyle='--', linewidth=0.5)
axes[1].set_xlabel('λ')
axes[1].set_ylabel('係数')
axes[1].set_title('Group Lasso: 同じ電極の係数がまとまって入る')
axes[1].invert_xaxis()
axes[1].legend(loc='upper left')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("図の見方:")
print("- 横軸は右→左でλが小さくなる（正則化が弱まる）")
print("- Lasso: 係数が個別に入る")
print("- Group Lasso: 同じ色（=同じ電極）の係数が同じλで同時に入る")

## 6. λ選択：交差検証

In [ ]:
from sklearn.model_selection import KFold
from sklearn.metrics import mean_squared_error

# 5-fold CV
kf = KFold(n_splits=5, shuffle=True, random_state=42)

cv_errors_lasso = []
cv_errors_glasso = []

for lam_lasso, lam_glasso in zip(lambdas_lasso, lambdas_glasso):
    errors_lasso = []
    errors_glasso = []
    
    for train_idx, val_idx in kf.split(X_scaled):
        X_train, X_val = X_scaled[train_idx], X_scaled[val_idx]
        y_train, y_val = y[train_idx], y[val_idx]
        
        # Lasso
        w_lasso = Lasso(alpha=lam_lasso, max_iter=10000).fit(X_train, y_train).coef_
        pred_lasso = X_val @ w_lasso
        errors_lasso.append(mean_squared_error(y_val, pred_lasso))
        
        # Group Lasso
        w_glasso = group_lasso_ista(X_train, y_train, groups, lam=lam_glasso)
        pred_glasso = X_val @ w_glasso
        errors_glasso.append(mean_squared_error(y_val, pred_glasso))
    
    cv_errors_lasso.append(np.mean(errors_lasso))
    cv_errors_glasso.append(np.mean(errors_glasso))

cv_errors_lasso = np.array(cv_errors_lasso)
cv_errors_glasso = np.array(cv_errors_glasso)

# 最適λ
best_idx_lasso = np.argmin(cv_errors_lasso)
best_idx_glasso = np.argmin(cv_errors_glasso)

print(f"最適λ（Lasso）: {lambdas_lasso[best_idx_lasso]:.4f}, CV MSE: {cv_errors_lasso[best_idx_lasso]:.4f}")
print(f"最適λ（Group Lasso）: {lambdas_glasso[best_idx_glasso]:.4f}, CV MSE: {cv_errors_glasso[best_idx_glasso]:.4f}")

# 可視化
fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(lambdas_lasso, cv_errors_lasso, 'o-', label='Lasso', alpha=0.7)
ax.plot(lambdas_glasso, cv_errors_glasso, 's-', label='Group Lasso', alpha=0.7)
ax.axvline(lambdas_lasso[best_idx_lasso], color='C0', linestyle='--', alpha=0.5)
ax.axvline(lambdas_glasso[best_idx_glasso], color='C1', linestyle='--', alpha=0.5)
ax.set_xscale('log')
ax.set_xlabel('λ')
ax.set_ylabel('CV Mean Squared Error')
ax.set_title('交差検証による最適λの選択')
ax.legend()
ax.grid(True, alpha=0.3)
plt.show()

## 7. 確認問題

**Q1. Lassoでは係数がちょうど0になるのに，Ridgeではならないのはなぜか**

→ L1は0で角を持ち，更新が軟閾値$S_\lambda(z)$になるので$|z| \le \lambda$の係数は0になる．Ridgeは$z/(1+\lambda)$と比例して縮むだけで0にはならない．

**Q2. 全グループの大きさが1のグループラッソは何になるか**

→ Lasso．$p_g = 1$で$\sqrt{p_g}\|w_g\|_2 = |w_j|$なので，罰則が$\lambda\|w\|_1$に一致する．

**Q3. 電極を選びつつ，残った電極の中で帯域も絞りたい．どの手法とグループの切り方を使うか**

→ Sparse Group Lassoで，グループ=1電極の全帯域．グループ項が電極を選び，L1項が残った電極の中の帯域を0にする．

## まとめ・自分の研究への含意

**利点**:
- Lasso: 特徴選択と予測を同時に行える凸最適化，解釈性が高い
- グループラッソ: 電極・帯域などの**構造を考慮した選択**が可能，グループ単位で0になるため解釈が明確
- 実装が容易（scikit-learn，または近接勾配法で数十行）

**限界**:
- 相関の強い特徴があるとLassoは恣意的に1つを選ぶ（Elastic Netで緩和）
- グループラッソはグループ内を疎にしない（Sparse Group Lassoが必要）
- 線形モデルなので非線形関係は捉えられない

**ECoG音声デコードへの応用**:
1. **電極選択**: 1電極の全帯域・全時点をグループとしてGroup Lassoで選択 → 重要な電極を特定
2. **帯域選択**: 1帯域の全電極をグループとして選択 → 低周波とハイガンマのどちらが効くかを判別
3. **同時選択**: Sparse Group Lassoで電極選択＋帯域選択を同時実行
4. **ニューラルネットとの併用**: 深層モデルの第1層にGroup L2罰則をかけて入力チャネル選択
5. **CoSAformerとの比較**: グループラッソで残る電極と，座標ベース注意重みが大きい電極の一致度を検証 → 注意機構の解釈の裏付け

**次に学ぶべきこと**:
- Sparse Group Lasso，Elastic Net，Fused Lassoの実装と使い分け
- FISTA（高速化版ISTA）の実装
- ECoGデータでの実験（電極配置・帯域分解・時間窓の構造化）
- ニューラルネットへのグループ正則化の組み込み